In [ ]:
import os
import numpy as np
import torch
import random
from ray import tune
from ray.rllib.algorithms.ppo import PPOConfig
from multiagent_ppo import MultiAgentF110, get_env_config, setup_policies_and_config
from ray.tune.registry import register_env
from ray.rllib.policy.policy import PolicySpec

# Set seeds for determinism
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)

# Register the environment
register_env("f1tenth_multi", lambda config: MultiAgentF110(config))

# Create temporary environment to get spaces and agents
temp_env = MultiAgentF110(get_env_config())
policies = {agent: PolicySpec(None, temp_env.observation_space, temp_env.action_space, {}) 
            for agent in temp_env.agents}
temp_env.close()

# Configure PPO for multi-agent training with determinism
config = (PPOConfig()
          .environment("f1tenth_multi", env_config=get_env_config())
          .framework("torch")
          .api_stack(enable_rl_module_and_learner=False, enable_env_runner_and_connector_v2=False)
          .env_runners(
              num_env_runners=0, # Only use one environment runner
              num_envs_per_env_runner=1,  # Ensure single environment per worker
          )
          .multi_agent(
              policies=policies, 
              policy_mapping_fn=lambda agent_id, *args, **kwargs: agent_id
          )
          .training(
              train_batch_size=200
          )
          .evaluation(
              evaluation_interval=10,
              evaluation_num_env_runners=1,
              evaluation_config={
                  "seed": SEED + 1  # Different seed for evaluation
              }
          )
          .debugging(
              seed=SEED  # Additional seed setting
          ))

# Run training with Ray Tune
tune.run(
    "PPO",
    config=config.to_dict(),
    stop={"timesteps_total": 20000},  # Train for 20,000 timesteps
    checkpoint_freq=10,  # Save checkpoint every 10 iterations
    storage_path=os.path.abspath("./ray_results"),  # Use absolute path
    name="f1tenth_multiagent_ppo",  # Experiment name 
)

2025-06-22 12:56:55,683	INFO worker.py:1888 -- Started a local Ray instance.
2025-06-22 12:56:56,293	INFO tune.py:253 -- Initializing Ray automatically. For cluster usage or custom Ray initialization, call `ray.init(...)` before `tune.run(...)`.
2025-06-22 12:56:56,295	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949
/home/sergio/.pyenv/versions/f1tenth/lib/python3.9/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/sergio/.pyenv/versions/f1tenth/lib/python3.9/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/home/sergio/.pyenv/versions/f1tenth/lib/python3.9/site-packages/gymnasium/utils/passi

(PPO pid=14083) Install gputil for GPU system monitoring.
(PPO pid=14083) /home/sergio/.pyenv/versions/f1tenth/lib/python3.9/site-packages/gymnasium/spaces/box.py:418: UserWarning: WARN: Casting input x to numpy array.
(PPO pid=14083)   gym.logger.warn("Casting input x to numpy array.")
(PPO pid=14083) 2025-06-22 12:57:06,300	WARNING deprecation.py:50 -- DeprecationWarning: `_get_slice_indices` has been deprecated. This will raise an error in the future!


Trial name,actor_manager_num_outstanding_async_reqs,agent_timesteps_total,counters,custom_metrics,episode_media,evaluation,info,num_agent_steps_sampled,num_agent_steps_sampled_lifetime,num_agent_steps_trained,num_env_steps_sampled,num_env_steps_sampled_lifetime,num_env_steps_sampled_this_iter,num_env_steps_sampled_throughput_per_sec,num_env_steps_trained,num_env_steps_trained_this_iter,num_env_steps_trained_throughput_per_sec,num_healthy_workers,num_remote_worker_restarts,num_steps_trained_this_iter,perf,timers
PPO_f1tenth_multi_9cff8_00000,0,35550,"{'num_env_steps_sampled': 20000, 'num_env_steps_trained': 20000, 'num_agent_steps_sampled': 35550, 'num_agent_steps_trained': 35550, 'num_env_steps_sampled_for_evaluation_this_iter': 3756}",{},{},"{'env_runners': {'episode_reward_max': -3.4587022567054078, 'episode_reward_min': -119.42746950740234, 'episode_reward_mean': -77.89800719928219, 'episode_len_mean': 375.6, 'episode_media': {}, 'episodes_timesteps_total': 3756, 'policy_reward_min': {'agent_0': -65.83884115410805, 'agent_1': -77.00533718595794}, 'policy_reward_max': {'agent_0': -0.232018287625408, 'agent_1': 22.918357280331747}, 'policy_reward_mean': {'agent_0': -37.95502807682813, 'agent_1': -39.94297912245409}, 'custom_metrics': {}, 'hist_stats': {'episode_reward': [-48.8614433927606, -101.94435972556232, -95.04178150086341, -99.43229881085881, -119.42746950740234, -62.55908943757376, -94.5892982043895, -97.16173439688085, -56.50389475982491, -3.4587022567054078], 'episode_lengths': [412, 199, 224, 284, 331, 627, 421, 191, 361, 706], 'policy_agent_0_reward': [-40.250165627859346, -38.44444906386594, -49.152803955798205, -47.12127088936871, -42.422132321444415, -0.232018287625408, -65.83884115410805, -35.01571904233016, -34.695820888843826, -26.37705953703727], 'policy_agent_1_reward': [-8.611277764901374, -63.4999106616964, -45.88897754506519, -52.311027921490066, -77.00533718595794, -62.32707114994836, -28.750457050281454, -62.14601535455067, -21.808073870981218, 22.918357280331747]}, 'sampler_perf': {'mean_raw_obs_processing_ms': 0.17536255130042983, 'mean_inference_ms': 0.7982521737610513, 'mean_action_processing_ms': 0.0950856488534415, 'mean_env_wait_ms': 0.5132265470250699, 'mean_env_render_ms': 0.0}, 'num_faulty_episodes': 0, 'connector_metrics': {'ObsPreprocessorConnector_ms': 0.017775297164916992, 'StateBufferConnector_ms': 0.0019431114196777344, 'ViewRequirementAgentConnector_ms': 0.04394173622131348}, 'num_episodes': 10, 'episode_return_max': -3.4587022567054078, 'episode_return_min': -119.42746950740234, 'episode_return_mean': -77.89800719928219, 'episodes_this_iter': 10}, 'num_agent_steps_sampled_this_iter': 5614, 'num_env_steps_sampled_this_iter': 3756, 'timesteps_this_iter': 3756, 'num_healthy_workers': 1, 'actor_manager_num_outstanding_async_reqs': 0, 'num_remote_worker_restarts': 0}","{'learner': {'agent_1': {'learner_stats': {'allreduce_latency': 0.0, 'grad_gnorm': 4.163370770215988, 'cur_kl_coeff': 0.6935440510448646, 'cur_lr': 5.0000000000000016e-05, 'total_loss': 5.645232057571411, 'policy_loss': -0.0023451012171184023, 'vf_loss': 5.645861697196961, 'vf_explained_var': -0.00853923757870992, 'kl': 0.00247353866604251, 'entropy': 5.310675303141276, 'entropy_coeff': 0.0}, 'model': {}, 'custom_metrics': {}, 'num_agent_steps_trained': 100.0, 'num_grad_updates_lifetime': 5430.5, 'diff_num_grad_updates_vs_sampler_policy': 29.5}, 'agent_0': {'learner_stats': {'allreduce_latency': 0.0, 'grad_gnorm': 12.366579143206279, 'cur_kl_coeff': 0.5773015156388285, 'cur_lr': 5.0000000000000016e-05, 'total_loss': 7.713785084088643, 'policy_loss': -0.012343171587175069, 'vf_loss': 7.719901140530904, 'vf_explained_var': -0.0016633431116739909, 'kl': 0.01078644848718616, 'entropy': 7.305025243759156, 'entropy_coeff': 0.0}, 'model': {}, 'custom_metrics': {}, 'num_agent_steps_trained': 87.0, 'num_grad_updates_lifetime': 5790.5, 'diff_num_grad_updates_vs_sampler_policy': 29.5}}, 'num_env_steps_sampled': 20000, 'num_env_steps_t

(RolloutWorker pid=14187) /home/sergio/.pyenv/versions/f1tenth/lib/python3.9/site-packages/gymnasium/spaces/box.py:418: UserWarning: WARN: Casting input x to numpy array.
(RolloutWorker pid=14187)   gym.logger.warn("Casting input x to numpy array.")
(PPO pid=14083) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/sergio/repos/f1_tenth_new_integration/examples/ray_results/f1tenth_multiagent_ppo/PPO_f1tenth_multi_9cff8_00000_0_2025-06-22_12-56-56/checkpoint_000000)
(RolloutWorker pid=14187) /home/sergio/.pyenv/versions/f1tenth/lib/python3.9/site-packages/gymnasium/spaces/box.py:418: UserWarning: WARN: Casting input x to numpy array.
(RolloutWorker pid=14187)   gym.logger.warn("Casting input x to numpy array.")
(PPO pid=14083) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/sergio/repos/f1_tenth_new_integration/examples/ray_results/f1tenth_multiagent_ppo/PPO_f1tenth_multi_9cff8_00000_0_2025-06-22_12-56-56/checkpoint_000001)
(RolloutWo

In [2]:
from multiagent_ppo import MultiAgentF110